<a href="https://colab.research.google.com/github/unmtransinfo/drugcentral-tools/blob/master/python/colab/DrugCentral_Inxight_Targets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DrugCentral - Inxight Integration - TARGETS

Merging DC and Inxight targets via UNIPROT IDs.

Note, from Inxight website: "The full database contains approximately 4,500 drugs, including FDA-approved, previously approved, over-the-counter, and investigational small-molecule and peptide drugs." Thus we expect many Inxight drugs will not be present in DrugCentral, which is scoped specifically with
*only* approved drugs.

*   Inxight data downloaded from: https://drugs.ncats.io/downloads-public
*   DrugCentral data available from: https://drugcentral.org/download


## Targets

Note that DrugCentral targets may be multi-component, single proteins as components. Thus Inxight targets are mapped to DC components.

In [18]:
import sys,os,re
import numpy as np
!pip install --upgrade pandas>=3.0.0
import pandas as pd
from google.colab import drive as colab_drive, auth as colab_auth
import gspread
from google.auth import default as g_auth_default

### Mount Google Drive

In [2]:
colab_drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATADIR='/content/drive/My Drive/UNM/DrugCentral/data/'
print(f'Files in {DATADIR}')
for dirname, _, filenames in os.walk(f'{DATADIR}'):
  for filename in filenames:
    print(os.path.join(dirname, filename))

Files in /content/drive/My Drive/UNM/DrugCentral/data/
/content/drive/My Drive/UNM/DrugCentral/data/dc2023_structures.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc2023_targets.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/Inxight Metadata.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/Inxight_Db.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/inxight_activity_targets_drugcentral_candidate.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_target_mapped.tsv
/content/drive/My Drive/UNM/DrugCentral/data/inx_act_mapped.tsv
/content/drive/My Drive/UNM/DrugCentral/data/inx_act_mapped.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_target_mapped.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_drug_mapped.tsv


### Authenticate with Google Colab for Google Sheets API access

In [4]:
colab_auth.authenticate_user()
creds, _ = g_auth_default()
gc = gspread.authorize(creds)

### Read data from Google Sheets

*   Inxight Targets
*   DrugCentral Targets



In [5]:
inx_sheet_url = 'https://docs.google.com/spreadsheets/d/1hqsjzcWH4m3SbLWoWga8O9IBmRmsI1tl62maItSIbN4/edit'
inx_target = None;
try:
    inx_ss = gc.open_by_url(inx_sheet_url) # Open spreadsheet by URL
    inx_ws_target = inx_ss.worksheet("frdbmeta-targets") # Select worksheets (specify or use get_worksheet(index))
    inx_target = inx_ws_target.get_all_values() # Get all values (list of lists)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Spreadsheet not found at URL: {inx_sheet_url}. Please check the URL and permissions.")
except gspread.exceptions.APIError as e:
    print(f"Error accessing Google Sheets API: {e}. Check permissions and ensure the sheet is shared correctly.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [19]:
df_inx_target = pd.DataFrame(inx_target[1:], columns=inx_target[0])
df_inx_target.drop(columns=['task', 'ddi_role', 'related_targets', 'synonym', 'human_ortholog'], inplace=True)
for tag in df_inx_target.columns:
  df_inx_target.rename(columns={tag: re.sub(r'^', 'inx_', tag)}, inplace=True)
df_inx_target['inx_UNIPROT'] = df_inx_target['inx_UNIPROT'].replace(r'^\s*$', np.nan)
print(f"Inxight Targets: {df_inx_target.shape[0]}; UNIPROTs: {df_inx_target['inx_UNIPROT'].nunique()}")
display(df_inx_target.sample(10))

Inxight Targets: 1295; UNIPROTs: 709


,inx_target,inx_UNIPROT,inx_class,inx_ChEMBL,inx_name,inx_organism
649,mitochondrial ferrochelatase,P22830,protein,CHEMBL3879831,"Ferrochelatase, mitochondrial",Homo sapiens
639,Mdr1a/b,-,group,,ATP-dependent translocase ABCB1,Mus musculus
1108,SULT1E1,P49888,protein,CHEMBL2346,Sulfotransferase 1E1,Homo sapiens
521,K2P16.1,Q96T55,protein,,Potassium channel subfamily K member 16,Homo sapiens
942,rCAT1,Q8R5I1,protein,,Cationic amino acid transporter-1 uORF,Rattus norvegicus
760,OAPT1B3,Q9NPD5,protein,,,Homo sapiens
664,mOATP1A4,Q9EP96,protein,CHEMBL2073700,Solute carrier organic anion transporter famil...,Mus musculus
76,ALDH1,P00352,protein,CHEMBL3577,Retinal dehydrogenase 1,Homo sapiens
672,monophosphate kinase,P30085,protein,CHEMBL5681,UMP-CMP kinase,Homo sapiens
610,MATE4,-,error,,,


In [20]:
display(df_inx_target['inx_class'].value_counts(sort=True))

inx_class
protein    793
group      245
error      144
synonym     77
variant     23
isoform      9
complex      4
Name: count, dtype: int64

In [21]:
display(df_inx_target['inx_organism'].value_counts(sort=True))

inx_organism
Homo sapiens                                                                                                                     820
                                                                                                                                 223
Rattus norvegicus                                                                                                                155
Mus musculus                                                                                                                      51
Sus scrofa                                                                                                                        13
Oryctolagus cuniculus                                                                                                              8
Bos taurus                                                                                                                         5
Ovis aries                                              

### Filter to keep only proteins.

In [22]:
df_inx_target = df_inx_target[df_inx_target['inx_class'].str.match('protein')]
print(f"Inx Targets, proteins, UNIPROTs: {df_inx_target['inx_UNIPROT'].nunique(dropna=True)}")
df_inx_target.drop(columns=['inx_class'], inplace=True)
display(df_inx_target.sample(10))

Inx Targets, proteins, UNIPROTs: 702


,inx_target,inx_UNIPROT,inx_ChEMBL,inx_name,inx_organism
998,rKCa1.1,Q62976,,,Rattus norvegicus
661,mOat2,Q91WU2,,,Mus musculus
839,P2X2,Q9UBL9,CHEMBL2531,P2X purinoceptor 2,Homo sapiens
1018,rOAT1,O35956,CHEMBL1777665,Solute carrier family 22 member 6,Rattus norvegicus
579,LPH,P09848,CHEMBL1075131,Lactase-phlorizin hydrolase,Homo sapiens
1147,Tx synthase,P24557,CHEMBL1835,Thromboxane-A synthase,Homo sapiens
24,ABCA3,Q99758,,Phospholipid-transporting ATPase ABCA3,Homo sapiens
1084,SQLE,Q14534,,,Homo sapiens
217,CYP219,P33261,,,Homo sapiens
318,CYP4F11,Q9HBI6,CHEMBL4295949,Cytochrome P450 4F11,Homo sapiens


### Read DC Targets file.



In [10]:
dc_sheet_target_url = 'https://docs.google.com/spreadsheets/d/1dCnTXIM2C8AOuTVRytWKpJk0FeS9PbW2mneXaWJ6fuc/edit'
dc_target = None;
try:
    dc_target_ss = gc.open_by_url(dc_sheet_target_url) # Open spreadsheet by URL
    dc_ws_target = dc_target_ss.worksheet("drugcentral_targets") # Select worksheets (specify or use get_worksheet(index))
    dc_target = dc_ws_target.get_all_values() # Get all values (list of lists)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Spreadsheet not found at URL: {dc_sheet_target_url}. Please check the URL and permissions.")
except gspread.exceptions.APIError as e:
    print(f"Error accessing Google Sheets API: {e}. Check permissions and ensure the sheet is shared correctly.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [23]:
df_dc_target = pd.DataFrame(dc_target[1:], columns=dc_target[0])
print(f"DC Targets: {df_dc_target.shape}")
print(f"DC IDs: {df_dc_target['target_id'].nunique()}; components: {df_dc_target['component_id'].nunique()}")
print(f"DC Target UNIPROTs: {df_dc_target['target_uniprot'].nunique()}")
df_dc_target.drop(columns=['target_id', 'target_name', 'protein_type', 'protein_components', 'swissprot'], inplace=True)

for tag in df_dc_target.columns:
  df_dc_target.rename(columns={tag: re.sub(r'^', 'dc_', tag)}, inplace=True)

display(df_dc_target.sample(10))

DC Targets: (4167, 13)
DC IDs: 3418; components: 3406
DC Target UNIPROTs: 3406


,dc_target_class,dc_component_id,dc_target_uniprot,dc_target_organism,dc_component_name,dc_gene_symbol,dc_geneid,dc_tdl
2126,Ion channel,5,A2AMW3,Mus musculus,Protein Gabre,Gabre,,
1891,GPCR,1654,Q03431,Homo sapiens,Parathyroid hormone/parathyroid hormone-relate...,PTH1R,5745,Tclin
1945,Ion channel,202,P18507,Homo sapiens,Gamma-aminobutyric acid receptor subunit gamma-2,GABRG2,2566,Tclin
1557,Enzyme,2445,P11541,Bos taurus,"Rod cGMP-specific 3',5'-cyclic phosphodiestera...",PDE6A,,
171,Enzyme,957,P18405,Homo sapiens,3-oxo-5-alpha-steroid 4-dehydrogenase 1,SRD5A1,6715,Tclin
3686,Nuclear other,535,P03372,Homo sapiens,Estrogen receptor,ESR1,2099,Tclin
2697,Transporter,13489,Q14973,Homo sapiens,Sodium/bile acid cotransporter,SLC10A1,6554,Tchem
3279,Ion channel,22685,Q63734,Rattus norvegicus,Potassium voltage-gated channel subfamily C me...,Kcnc4,,
3370,Transporter,13485,Q12908,Homo sapiens,Ileal sodium/bile acid cotransporter,SLC10A2,6555,Tclin
2499,GPCR,343,P42866,Mus musculus,Mu-type opioid receptor,Oprm1,,


### Merge Inxight targets with DC via UNIPROT IDs.

In [24]:
dc_inx_target = pd.merge(df_dc_target, df_inx_target, left_on='dc_target_uniprot', right_on='inx_UNIPROT', how='inner')
dc_inx_target = dc_inx_target[dc_inx_target['dc_target_uniprot'].notna() & dc_inx_target['inx_UNIPROT'].notna()]
dc_inx_target.drop_duplicates(inplace=True)
display(dc_inx_target.head(10))

,dc_target_class,dc_component_id,dc_target_uniprot,dc_target_organism,dc_component_name,dc_gene_symbol,dc_geneid,dc_tdl,inx_target,inx_UNIPROT,inx_ChEMBL,inx_name,inx_organism
0,Enzyme,2201,Q9UGN5,Homo sapiens,Poly [ADP-ribose] polymerase 2,PARP2,10038,Tclin,PARP2,Q9UGN5,,,Homo sapiens
1,Transporter,194,O00337,Homo sapiens,Sodium/nucleoside cotransporter 1,SLC28A1,9154,Tbio,CNT1,O00337,CHEMBL5551,Sodium/nucleoside cotransporter 1,Homo sapiens
2,Transporter,1214,P31639,Homo sapiens,Sodium/glucose cotransporter 2,SLC5A2,6524,Tclin,SGLT2,P31639,CHEMBL3884,Sodium/glucose cotransporter 2,Homo sapiens
3,Transporter,1214,P31639,Homo sapiens,Sodium/glucose cotransporter 2,SLC5A2,6524,Tclin,SLGT2,P31639,,,Homo sapiens
4,Kinase,1058,P52333,Homo sapiens,Tyrosine-protein kinase JAK3,JAK3,3718,Tclin,JAK3,P52333,,,Homo sapiens
5,Enzyme,2617,P35354,Homo sapiens,Prostaglandin G/H synthase 2,PTGS2,5743,Tclin,PGHS-2,P35354,,,Homo sapiens
6,Ion channel,1962,Q8NER1,Homo sapiens,Transient receptor potential cation channel su...,TRPV1,7442,Tclin,TRPV1,Q8NER1,CHEMBL4794,Transient receptor potential cation channel su...,Homo sapiens
7,GPCR,1008,P21554,Homo sapiens,Cannabinoid receptor 1,CNR1,1268,Tclin,CB1,P21554,,,Homo sapiens
8,Ion channel,1728,Q14524,Homo sapiens,Sodium channel protein type 5 subunit alpha,SCN5A,6331,Tclin,Nav1.5,Q14524,CHEMBL1980,Sodium channel protein type 5 subunit alpha,Homo sapiens
9,Enzyme,2616,P23219,Homo sapiens,Prostaglandin G/H synthase 1,PTGS1,5742,Tclin,COX1,P23219,CHEMBL221,Prostaglandin G/H synthase 1,Homo sapiens


### Export mapped UniProts

In [25]:
dc_inx_target.to_csv(f"{DATADIR}/dc_inx_target_mapped.tsv", sep='\t', index=False)